In [1]:
from ingest import load_faq_data, build_index
documents = load_faq_data()

In [2]:
## call LLM for 1 document:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
# Wrap the search call in a function for text search
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


In [6]:
# Start with one ground truth
q = ground_truth[0]
q

{'question': 'I just found this course — am I still allowed to join, or is it too late?',
 'document': '74eb249bbf'}

In [7]:
# Run text search for this query
doc_id = q["document"]
results = text_search(query=q["question"])

In [8]:
# Compare retrieve document id with the correct document id:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
c2903069a0 == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
0fab61eca2 == 74eb249bbf: False


In [9]:
# turn this comparison into a relevance list with 0's and 1's:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [10]:
# turn relevance calculation into a function:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [11]:
# compute relevance for the first ground truth:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course — am I still allowed to join, or is it too late?


[1, 0, 0, 0, 0]

In [12]:
# 2nd example:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

How can I ask questions during the live sessions if I’m not in the Zoom room?


[1, 0, 0, 0, 0]

In [13]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I check the LLM Zoomcamp syllabus and see what’s in the current cohort?


[1, 0, 0, 0, 0]

In [14]:
# Let's repeat the process for all ground truths:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [15]:
# call first 15 ground truths:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [16]:
# explore relevance total text:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [17]:
# make the relevance functions generic by either passing text, or vector, or hybrid search
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [18]:
# total relevance of all ground truths also become generic
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [25]:
## App;y text search
relevance_total_sample = compute_relevance_total(ground_truth_sample, text_search)
relevance_total_sample

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [21]:
# run it for all ground truth questions:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

In [24]:
relevance_total

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,

In [26]:
# Hit Rate and MRR
# Hit Rate: measures the fraction of queries where the correct document appears anywhere in the results
cnt = 0

for line in relevance_total_sample:
    if 1 in line:
        cnt = cnt + 1

cnt

12

In [27]:
cnt / len(relevance_total_sample)

0.8

In [28]:
# let's put hit rate to a function:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [29]:
hit_rate(relevance_total_sample)
# 0.933

0.8

In [ ]:
# MRR: aka Mean Reciprocal Rank, tells us if we found the right document and at which position
relevance_total_sample[1] # 2nd index has the answer, so the score becomes 1/2

[0, 1, 0, 0, 0]

In [31]:
# let's calculate MRR:
total_score = 0.0

for line in relevance_total_sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

9.666666666666666

In [32]:
#Divide it by the number of queries:
total_score / len(relevance_total_sample)



0.6444444444444444

In [35]:
#MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.
#Hit Rate is the upper bound for MRR. 
#In practice, MRR is usually smaller because some correct documents are found below the first position.

In [36]:
# put MRR into a function:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [37]:
mrr(relevance_total_sample)

0.6444444444444444

In [38]:
## Putting all metrics together: computer relevance and get both - hit rate and mrr
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [39]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8442477876106195, 'mrr': 0.7143067846607668}

In [41]:
#Search metrics tell us whether retrieval works. Next, we'll use these metrics to tune the search parameters.
#  A high MRR means the relevant document is near the top, which helps the LLM focus on the right context. 
# A low MRR with a high hit rate means the document is there, but buried under less relevant results.